# Book Researcher: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [3_crew/community_contributions/Book_Researcher/pyproject.toml](../../../../../../3_crew/community_contributions/Book_Researcher/pyproject.toml)
- [3_crew/community_contributions/Book_Researcher/README.md](../../../../../../3_crew/community_contributions/Book_Researcher/README.md)
- [3_crew/community_contributions/Book_Researcher/src/book/__init__.py](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/__init__.py)
- [3_crew/community_contributions/Book_Researcher/src/book/config/agents.yaml](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/config/agents.yaml)
- [3_crew/community_contributions/Book_Researcher/src/book/config/tasks.yaml](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/config/tasks.yaml)
- [3_crew/community_contributions/Book_Researcher/src/book/crew.py](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/crew.py)
- [3_crew/community_contributions/Book_Researcher/src/book/main.py](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/main.py)
- [3_crew/community_contributions/Book_Researcher/src/book/tools/__init__.py](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/tools/__init__.py)
- [3_crew/community_contributions/Book_Researcher/src/book/tools/custom_tool.py](../../../../../../3_crew/community_contributions/Book_Researcher/src/book/tools/custom_tool.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 4. Plan → search → synthesize

The research workflow separates a search plan, collected findings, and a report. Use fixed fictional snippets first so you can inspect how evidence reaches the writer. No current facts are being researched here.

**Predict:** Will the report receive the original query as well as the findings?

In [ ]:
from dataclasses import dataclass
@dataclass
class SearchItem:
    reason: str
    query: str
query = "How should a beginner practice tool calls?"
plan = [SearchItem("Understand arguments", "tool argument schema"),
        SearchItem("Understand execution", "tool dispatcher")]
fixture = {"tool argument schema": "A schema describes arguments.",
           "tool dispatcher": "Application code executes an allowed function."}
findings = [fixture[item.query] for item in plan]
report = {"query": query, "summary": " ".join(findings), "follow_up_questions": ["How do we test invalid arguments?"]}
print(report)
assert len(findings) == 2 and "Application" in report["summary"]

**Check:** Yes. Plans choose questions, search collects material, and synthesis uses it. Delivery is a separate side effect.

**Change one thing:** Add a third search plan item and a matching fixture. Explain how the original query keeps synthesis focused.

## 5. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 6. Roles, tasks, and process

CrewAI separates who does work from what work is required. A sequential process has a fixed task order; a hierarchical process gives a manager coordination responsibility.

**Predict:** Is a role description itself an executable task?

In [ ]:
agents = {"researcher": "Collect fictional evidence", "analyst": "Interpret supplied evidence"}
tasks = [{"name": "research_task", "agent": "researcher", "expected_output": "Findings"},
         {"name": "analysis_task", "agent": "analyst", "expected_output": "Short report"}]
outputs = []
for task in tasks:
    assert task["agent"] in agents
    outputs.append({"task": task["name"], "input_from_previous": list(outputs), "result": task["expected_output"]})
print([item["task"] for item in outputs])
assert len(outputs[1]["input_from_previous"]) == 1

**Check:** No. A task assigns work to a role. This dictionary simulation does not execute CrewAI.

**Change one thing:** Add a reviewer role and task. Describe whether its input should include the analyst output.

## 7. Configuration fills named placeholders

The crew entry point supplies inputs, and YAML descriptions contain placeholders. Configuration names must match the inputs used at runtime.

**Predict:** What happens if motion is not supplied?

In [ ]:
description = "Propose an argument for: {motion}."
inputs = {"motion": "Beginners should practice offline first"}
rendered = description.format(**inputs)
print(rendered)
assert inputs["motion"] in rendered
try:
    description.format(**{})
except KeyError:
    print("Missing placeholder detected")

**Check:** Formatting raises KeyError here. Crew configuration and project environment are separate from the root project.

**Change one thing:** Add a second placeholder named audience and provide its value. Identify matching input keys in the linked source main.py.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.